# Performance Optimization & Query Analysis

## Using Snowflake's built-in insights to find and fix performance issues

In this lab you'll learn to analyze your own query history from yesterday's HOL to:
- Identify your slowest and most expensive queries
- Understand warehouse utilization patterns
- Find optimization opportunities (spilling, pruning, caching)
- Apply practical fixes

**Prerequisites:** You must have run HOL 1 yesterday (so query history is populated).

---

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE BB_TRAINING_WH;
USE DATABASE BB_TRAINING;

---
## 1. Query History Overview

Let's look at everything that ran on your account. This uses `ACCOUNT_USAGE.QUERY_HISTORY` which has a ~45 minute latency but full detail.

In [ ]:
-- How many queries have you run? Break down by status and type
SELECT
    EXECUTION_STATUS,
    COUNT(*) AS query_count,
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 2) AS avg_duration_sec,
    ROUND(SUM(BYTES_SCANNED) / 1024 / 1024, 1) AS total_mb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
GROUP BY EXECUTION_STATUS
ORDER BY query_count DESC;

In [ ]:
-- Query volume by type (DDL, DML, SELECT)
SELECT
    QUERY_TYPE,
    COUNT(*) AS query_count,
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 2) AS avg_duration_sec,
    ROUND(MAX(TOTAL_ELAPSED_TIME) / 1000, 2) AS max_duration_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
  AND EXECUTION_STATUS = 'SUCCESS'
GROUP BY QUERY_TYPE
ORDER BY query_count DESC;

---
## 2. Find Your Slowest Queries

The first optimization step: identify what's taking the longest. In production, these are your targets for improvement.

In [ ]:
-- Top 10 slowest queries
SELECT
    QUERY_ID,
    QUERY_TYPE,
    ROUND(TOTAL_ELAPSED_TIME / 1000, 2) AS duration_sec,
    ROUND(COMPILATION_TIME / 1000, 2) AS compile_sec,
    ROUND(EXECUTION_TIME / 1000, 2) AS exec_sec,
    ROUND(BYTES_SCANNED / 1024 / 1024, 2) AS mb_scanned,
    ROWS_PRODUCED,
    PARTITIONS_SCANNED,
    PARTITIONS_TOTAL,
    LEFT(QUERY_TEXT, 80) AS query_preview
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
  AND EXECUTION_STATUS = 'SUCCESS'
ORDER BY TOTAL_ELAPSED_TIME DESC
LIMIT 10;

---
## 3. Compilation vs Execution Time

A common finding: queries spend more time compiling (planning) than executing. This happens with complex SQL or cold metadata caches.

In [ ]:
-- Where is time being spent: compilation vs execution vs queuing?
SELECT
    QUERY_TYPE,
    COUNT(*) AS queries,
    ROUND(AVG(COMPILATION_TIME) / 1000, 2) AS avg_compile_sec,
    ROUND(AVG(EXECUTION_TIME) / 1000, 2) AS avg_exec_sec,
    ROUND(AVG(QUEUED_OVERLOAD_TIME) / 1000, 2) AS avg_queue_sec,
    ROUND(AVG(COMPILATION_TIME) * 100.0 / NULLIF(AVG(TOTAL_ELAPSED_TIME), 0), 1) AS compile_pct
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
  AND EXECUTION_STATUS = 'SUCCESS'
  AND TOTAL_ELAPSED_TIME > 0
GROUP BY QUERY_TYPE
ORDER BY avg_compile_sec DESC;

---
## 4. Partition Pruning Efficiency

Partition pruning is Snowflake's primary optimization. When `PARTITIONS_SCANNED` is much less than `PARTITIONS_TOTAL`, Snowflake skipped irrelevant data. When they're equal, it did a full scan.

In [ ]:
-- Partition pruning analysis: how efficiently are queries scanning data?
SELECT
    QUERY_ID,
    PARTITIONS_SCANNED,
    PARTITIONS_TOTAL,
    ROUND(PARTITIONS_SCANNED * 100.0 / NULLIF(PARTITIONS_TOTAL, 0), 1) AS scan_pct,
    CASE
        WHEN PARTITIONS_TOTAL = 0 THEN 'N/A'
        WHEN PARTITIONS_SCANNED * 100.0 / PARTITIONS_TOTAL <= 20 THEN 'EXCELLENT pruning'
        WHEN PARTITIONS_SCANNED * 100.0 / PARTITIONS_TOTAL <= 50 THEN 'GOOD pruning'
        WHEN PARTITIONS_SCANNED * 100.0 / PARTITIONS_TOTAL <= 80 THEN 'MODERATE - consider clustering'
        ELSE 'FULL SCAN - needs optimization'
    END AS pruning_assessment,
    ROUND(BYTES_SCANNED / 1024 / 1024, 2) AS mb_scanned,
    LEFT(QUERY_TEXT, 60) AS query_preview
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
  AND EXECUTION_STATUS = 'SUCCESS'
  AND PARTITIONS_TOTAL > 0
ORDER BY scan_pct DESC
LIMIT 15;

---
## 7. Warehouse Utilization & Credit Consumption

How efficiently are you using your warehouse? Long suspend times waste credits, but too-short auto-suspend means frequent cold starts.

In [ ]:
-- Warehouse credit consumption over time
SELECT
    DATE_TRUNC('hour', START_TIME) AS hour,
    WAREHOUSE_NAME,
    ROUND(SUM(CREDITS_USED), 4) AS credits_used,
    COUNT(*) AS queries_run
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
GROUP BY hour, WAREHOUSE_NAME
ORDER BY hour;

In [ ]:
-- Warehouse sizing check: is XS enough or should we size up?
SELECT
    WAREHOUSE_SIZE,
    COUNT(*) AS total_queries,
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 2) AS avg_duration_sec,
    ROUND(MAX(TOTAL_ELAPSED_TIME) / 1000, 2) AS max_duration_sec,
    SUM(BYTES_SPILLED_TO_LOCAL_STORAGE) AS total_local_spill,
    SUM(BYTES_SPILLED_TO_REMOTE_STORAGE) AS total_remote_spill,
    CASE
        WHEN SUM(BYTES_SPILLED_TO_REMOTE_STORAGE) > 0 THEN 'SIZE UP - remote spilling detected'
        WHEN SUM(BYTES_SPILLED_TO_LOCAL_STORAGE) > 0 THEN 'CONSIDER sizing up - local spilling'
        WHEN MAX(TOTAL_ELAPSED_TIME) > 60000 THEN 'CONSIDER sizing up - queries > 60s'
        ELSE 'CURRENT SIZE OK'
    END AS sizing_recommendation
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND WAREHOUSE_NAME = 'BB_TRAINING_WH'
  AND EXECUTION_STATUS = 'SUCCESS'
GROUP BY WAREHOUSE_SIZE;

---
## 6. Query Insights (Snowflake's Built-in Recommendations)

Snowflake automatically analyzes your queries and produces **insights** - specific performance issues with actionable suggestions. This is the `QUERY_INSIGHTS` view: Snowflake tells you what's wrong and how to fix it.

In [ ]:
-- All query insights from the last 2 days
SELECT
    QUERY_ID,
    INSIGHT_TYPE_ID,
    INSIGHT_TOPIC,
    IS_OPPORTUNITY,
    MESSAGE,
    SUGGESTIONS,
    ROUND(TOTAL_ELAPSED_TIME / 1000, 2) AS duration_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_INSIGHTS
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
ORDER BY IS_OPPORTUNITY DESC, TOTAL_ELAPSED_TIME DESC;

In [ ]:
-- Summary: what types of issues are most common?
SELECT
    INSIGHT_TOPIC,
    INSIGHT_TYPE_ID,
    COUNT(*) AS occurrence_count,
    SUM(CASE WHEN IS_OPPORTUNITY THEN 1 ELSE 0 END) AS actionable_count
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_INSIGHTS
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
GROUP BY INSIGHT_TOPIC, INSIGHT_TYPE_ID
ORDER BY occurrence_count DESC;

In [ ]:
-- Actionable opportunities only - things Snowflake recommends you fix
SELECT
    QUERY_ID,
    INSIGHT_TOPIC,
    MESSAGE:"description"::STRING AS issue_description,
    SUGGESTIONS[0]::STRING AS recommendation,
    ROUND(TOTAL_ELAPSED_TIME / 1000, 2) AS duration_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_INSIGHTS
WHERE START_TIME >= DATEADD('day', -2, CURRENT_TIMESTAMP())
  AND IS_OPPORTUNITY = TRUE
ORDER BY TOTAL_ELAPSED_TIME DESC
LIMIT 10;

---
## Key Takeaways

| What to check | View / Tool | What it tells you |
|---|---|---|
| Slowest queries | `QUERY_HISTORY` | Which queries to optimize first |
| Compile vs execute time | `QUERY_HISTORY` | Whether SQL complexity or data volume is the bottleneck |
| Partition pruning | `QUERY_HISTORY` | Whether queries are scanning too much data |
| Warehouse utilization | `WAREHOUSE_METERING_HISTORY` | Whether your warehouse is the right size |
| Query Insights | `QUERY_INSIGHTS` | Snowflake's auto-detected issues with fix suggestions |

### The optimization workflow in production:
1. **Check Query Insights** first - Snowflake already found issues for you
2. **Identify top spenders** - slowest queries or most-scanned data
3. **Diagnose** - is it pruning, spilling, queueing, or compilation?
4. **Fix** - add filters, clustering keys, size up warehouse, or restructure SQL
5. **Verify** - re-run and compare before/after metrics